In [1]:
import pandas as pd
import os
from collections import defaultdict
from pprint import pprint
import re


In [2]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")
processed_dir = os.path.join(data_dir, "structured")

survey_filename = "survey_1.csv"
raw_path = os.path.join(raw_dir, survey_filename)
processed_path = os.path.join(processed_dir, survey_filename)


In [3]:
df = pd.read_csv(raw_path)

df["Orientación sexual"] = (
    df["Orientación Sexual"]
    .fillna(df["Orientación Sexual.1"])
)

meta_cols = {
    "Marca temporal": "timestamp",
    "Edad": "age",
    "Género": "gender",
    "Orientación sexual": "sexual_orientation",
}

drop_cols = [
    "Marca temporal",
    "Edad",
    "Género",
    "Orientación Sexual",
    "Orientación Sexual.1",
    "Orientación sexual",
]

meta = (
    df[list(meta_cols.keys())]
    .rename(columns=meta_cols)
    .copy()
)

data = df.drop(columns=drop_cols)

meta.head()


,timestamp,age,gender,sexual_orientation
0,18/04/2026 21:20:28,21,Femenino,Heterosexual
1,19/04/2026 15:15:37,24,Femenino,Homosexual
2,19/04/2026 15:44:41,23,Otros,Homosexual
3,19/04/2026 20:57:37,20,Otros,Homosexual
4,20/04/2026 15:15:28,23,Femenino,Homosexual


In [4]:
free_text_cols = [
	c for c in data.columns
	if "(1 frase)" in c
]

print(free_text_cols)


['Escribe tu respuesta según la opción elegida en la pregunta anterior (1 frase).', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase).', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..1', 'Escribe cómo saludarías a Pablo, tu amigo online que hiciste hace unas semanas (1 frase).', 'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 frase)..1', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..2', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..3', 'Escribe cómo saludarías a Lucía, tu amiga online que hiciste hace una semanas (1 frase).']


In [5]:
blocks = []
current_block = []

for col in data.columns:
	current_block.append(col)
	
	if col in free_text_cols:
		blocks.append(current_block)
		current_block = []
		
print(blocks)

[['¿Qué tono tendría tu respuesta?', 'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 frase).'], ['¿Cómo de preocupado estás por Pablo?', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase).'], ['¿Cómo te lo estás pasando?', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..1'], ['Escribe cómo saludarías a Pablo, tu amigo online que hiciste hace unas semanas (1 frase).'], ['¿Qué tono tendría tu respuesta?.1', 'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 frase)..1'], ['¿Cómo de preocupado estás por Lucía?', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..2'], ['¿Cómo te lo estás pasando?.1', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..3'], ['Escribe cómo saludarías a Lucía, tu amiga online que hiciste hace una semanas (1 frase).']]


In [6]:
n = len(blocks) // 2

first_half = blocks[:n]
second_half = blocks[n:]

print(first_half)

# print(second_half)

[['¿Qué tono tendría tu respuesta?', 'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 frase).'], ['¿Cómo de preocupado estás por Pablo?', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase).'], ['¿Cómo te lo estás pasando?', 'Escribe tu respuesta según la opción seleccionada en la pregunta anterior (1 frase)..1'], ['Escribe cómo saludarías a Pablo, tu amigo online que hiciste hace unas semanas (1 frase).']]


In [7]:
merged = defaultdict(list)
situations = [1, 4, 7, 10]

merged = {
    situation: [first, second]
    for situation, first, second in zip(
        situations,
        first_half,
        second_half,
    )
}
		

In [8]:
pprint(merged[1])


[['¿Qué tono tendría tu respuesta?',
  'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 '
  'frase).'],
 ['¿Qué tono tendría tu respuesta?.1',
  'Escribe tu respuesta según la opción elegida en la pregunta anterior (1 '
  'frase)..1']]


In [9]:
rows = []

for i, survey_row in df.iterrows():
    row = {
        col: meta.loc[i, col]
        for col in meta.columns
    }

    for idx, pairs in merged.items():
        for columns in pairs:

            labels = ["text"] if len(columns) == 1 else ["choice", "text"]

            for column, label in zip(columns, labels):
                value = survey_row[column]

                if pd.notna(value):
                    row[f"situation_{idx}_{label}"] = value
                    
    rows.append(row)

clean_df = pd.DataFrame(rows)
clean_df.head()

,timestamp,age,gender,sexual_orientation,situation_1_choice,situation_1_text,situation_4_choice,situation_4_text,situation_7_choice,situation_7_text,situation_10_text
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,Amigable y cordial.,"“Hola, soy Patri! Encantada de conocerte”","Preocupado, mostrando interés por cómo se encu...",“¿Enserio? ¿Pero te has tomado algo para el do...,"Bien, divirtiéndote y disfrutando del momento.","""Super bien! La verdad es que todos son bastan...","""Heyy sigues despierto? Estás haciendo algo ah..."
1,19/04/2026 15:15:37,24,Femenino,Homosexual,Amigable y cordial.,"Holaaa buenas, soy Nombre, acabo de llegar y e...","Indiferente, sin mostrar especial interés.","Ayyy qué rollo, bueno al menos ya estás bien","Más o menos, no del todo cómodo.",no aguanto más me quiero ir xd,"volví de la fiesta, sigues por ahí?"
2,19/04/2026 15:44:41,23,Otros,Homosexual,Tímido y reservado.,"Hola, Laura, encantado.","Preocupado, mostrando interés por cómo se encu...","Ay, estas bien?","Más o menos, no del todo cómodo.",No muy bien. No conozco a casi nadie y no sé q...,"Holi, ya he vuelto de la fiesta, qué tal te ha..."
3,19/04/2026 20:57:37,20,Otros,Homosexual,Tímido y reservado.,Emmm. Hola.,"Preocupado, mostrando interés por cómo se encu...",oooo Todo guay seguro?,"Bien, divirtiéndote y disfrutando del momento.",Guay la verdad q la gente es muy maja aqui,"Estoy en casa ya, que tal??"
4,20/04/2026 15:15:28,23,Femenino,Homosexual,Amigable y cordial.,Hola Laura yo soy Rocío encantada,"Preocupado, mostrando interés por cómo se encu...",Ay no pobre cómo estás ahora ?,"Bien, divirtiéndote y disfrutando del momento.",Muy bien!,Hola Lucía! Tú día qué tal ha ido ?


In [10]:
def clean_text(text):
	if pd.isna(text):
		return text

	out = str(text)

	out = re.sub(r"\(.*?\)", "", out)
	out = re.sub(r"\*.*?\*", "", out)
	out = re.sub(r"[\"“”]", "", out)
	out = re.sub(r"\s+", " ", out)
	out = out.strip()

	return out



In [11]:
text_columns = [
    column
    for column in clean_df.columns
    if "text" in column
]

clean_df[text_columns] = clean_df[text_columns].map(clean_text)

display(clean_df)

,timestamp,age,gender,sexual_orientation,situation_1_choice,situation_1_text,situation_4_choice,situation_4_text,situation_7_choice,situation_7_text,situation_10_text
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,Amigable y cordial.,"Hola, soy Patri! Encantada de conocerte","Preocupado, mostrando interés por cómo se encu...",¿Enserio? ¿Pero te has tomado algo para el dol...,"Bien, divirtiéndote y disfrutando del momento.",Super bien! La verdad es que todos son bastant...,Heyy sigues despierto? Estás haciendo algo ahora?
1,19/04/2026 15:15:37,24,Femenino,Homosexual,Amigable y cordial.,"Holaaa buenas, soy Nombre, acabo de llegar y e...","Indiferente, sin mostrar especial interés.","Ayyy qué rollo, bueno al menos ya estás bien","Más o menos, no del todo cómodo.",no aguanto más me quiero ir xd,"volví de la fiesta, sigues por ahí?"
2,19/04/2026 15:44:41,23,Otros,Homosexual,Tímido y reservado.,"Hola, Laura, encantado.","Preocupado, mostrando interés por cómo se encu...","Ay, estas bien?","Más o menos, no del todo cómodo.",No muy bien. No conozco a casi nadie y no sé q...,"Holi, ya he vuelto de la fiesta, qué tal te ha..."
3,19/04/2026 20:57:37,20,Otros,Homosexual,Tímido y reservado.,Emmm. Hola.,"Preocupado, mostrando interés por cómo se encu...",oooo Todo guay seguro?,"Bien, divirtiéndote y disfrutando del momento.",Guay la verdad q la gente es muy maja aqui,"Estoy en casa ya, que tal??"
4,20/04/2026 15:15:28,23,Femenino,Homosexual,Amigable y cordial.,Hola Laura yo soy Rocío encantada,"Preocupado, mostrando interés por cómo se encu...",Ay no pobre cómo estás ahora ?,"Bien, divirtiéndote y disfrutando del momento.",Muy bien!,Hola Lucía! Tú día qué tal ha ido ?
5,21/04/2026 15:19:26,24,Femenino,Bisexual,Amigable y cordial.,"Hol, soy Briana un gusto","Indiferente, sin mostrar especial interés.","Me alegra escuchar que ya te sientes mejor, el...","Más o menos, no del todo cómodo.","Bien, ósea recién estoy conociendo a la gente ...",Hey como estas ? Sigues despierto ?
6,4/05/2026 17:19:20,23,Masculino,Heterosexual,Tímido y reservado.,Hola,"Indiferente, sin mostrar especial interés.",,"Bien, divirtiéndote y disfrutando del momento.",Bien :),Hola
7,4/05/2026 18:35:52,18,Masculino,Heterosexual,Amigable y cordial.,"Buenas, yo soy Miguel, encantado","Preocupado, mostrando interés por cómo se encu...",Y eso? Que te pasa?,"Bien, divirtiéndote y disfrutando del momento.","Bien, aquí disfrutando con los amigos, tu que ...","Holaaa, que tall?"
8,4/05/2026 18:38:38,18,Masculino,Heterosexual,Amigable y cordial.,Encantado soy ...,"Preocupado, mostrando interés por cómo se encu...",Así? No es nada grave no?,"Más o menos, no del todo cómodo.","Bueno bien, no conozco muy bien a esta gente","Ya estoy en casa, tu que tal? Que haces?"
9,4/05/2026 19:26:02,22,Masculino,Heterosexual,Amigable y cordial.,"Hola, soy Gianfranco un placer","Preocupado, mostrando interés por cómo se encu...","Espero que te mejores pronto!, por lo menos no...","Bien, divirtiéndote y disfrutando del momento.",Súper bien!,Lucía! avisame cuando llegues a tu casa


In [12]:
clean_df.to_csv(processed_path, index=False)

situations_dir = os.path.join(processed_dir, "situations")
os.makedirs(situations_dir, exist_ok=True)

for s in situations:
    cols = [
        col for col in clean_df.columns
        if col.startswith(f"situation_{s}_")
    ]

    subdf = clean_df[
        ["timestamp", "age", "gender", "sexual_orientation", *cols]
    ].copy()

    subdf = subdf.rename(columns={
        f"situation_{s}_choice": "choice",
        f"situation_{s}_text": "text"
    })

    path = os.path.join(situations_dir, f"situation_{s}.csv")
    subdf.to_csv(path, index=False)
